# 심장 구조 MRI 분할 (ACDC) 불균형 세그멘테이션 실험

---

## 1. 태스크 및 도메인
- **도메인**: 심장 구조 (Cardiac Structure) MRI 세그멘테이션
- **모달리티**: Cardiac Cine MRI (그레이스케일)
- **태스크**: 4-class segmentation
  - 클래스: 배경(0) / 우심실 RV(1) / 심근 MYO(2) / 좌심실 LV(3)
- **핵심 도전**: RV/MYO 클래스 불균형(~5~15:1), ED/ES 두 시점의 심장 형태 변화, 심근의 얇은 경계

## 2. 모델
- **아키텍처**: U-Net (ResNet34 백본)
- **사전학습**: ImageNet pretrained
- **선택 이유**: Multi-class segmentation 표준 베이스라인, 다도메인 비교 일관성
- **출력**: 4채널 softmax 직접 출력 (multi-class — `to_2ch_logits()` 변환 불필요)
- **입력**: Grayscale → 3채널 복제 (ImageNet 인코더 3ch 맞춤)

## 3. 데이터셋
- **이름**: ACDC (Automated Cardiac Diagnosis Challenge)
- **규모**: 100명 환자 — Train 70 / Val 15 / Test 15 (환자 단위 분할)
  - 각 환자: ED(확장기) + ES(수축기) 두 시점, 시점당 다수 2D 슬라이스
- **입력 해상도**: 256×256 (리사이즈)
- **클래스 불균형**: BG:RV ≈ **~5~15:1**, BG:MYO ≈ **~5~10:1**, BG:LV ≈ **~3~8:1**
- **공식 분할**: 없음 → 환자 단위 7:1.5:1.5 랜덤 분할 (random_state=42)

## 4. 데이터 준비 (협업자용)
> Google Drive에 zip 파일을 업로드한 후 노트북을 실행할 것. Cell 0 실행 시 자동으로 압축 해제.

**취득 방법**:
- 공식 챌린지: https://www.creatis.insa-lyon.fr/Challenge/acdc/ (계정 등록 후 다운로드)
- 다운로드 후 zip 압축: `zip -r acdc.zip training/`

**Colab Drive 업로드 경로**:
```
MyDrive/imbalanced-data-LWCE/acdc/
  acdc.zip   ← training/{patient_id}/ 폴더 전체 압축
```
zip 내부 각 환자 폴더 구성:
- `{patient_id}_frame{ED:02d}.nii.gz` + `{patient_id}_frame{ED:02d}_gt.nii.gz`
- `{patient_id}_frame{ES:02d}.nii.gz` + `{patient_id}_frame{ES:02d}_gt.nii.gz`
- `Info_{patient_id}.cfg` (ED/ES 프레임 번호 포함)

## 5. 전처리 및 도메인 특이점
- Percentile 정규화 (1~99th, 비-zero 복셀 기준) — 볼륨(3D 프레임) 단위 적용
- Grayscale → 3채널 복제 (ImageNet 인코더 3ch 채널 수 맞춤)
- 환자 단위 분할 (슬라이스 단위 분할 시 동일 환자 슬라이스가 train/val에 섞여 data leakage 발생)
- ED/ES 두 시점만 사용 — 4D cine(`patient_4d.nii.gz`) 전체는 미사용
- ImageNet 정규화 (mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])

## 6. 실험 손실 함수 및 Optuna 탐색 범위
| 손실 함수 | 탐색 파라미터 | 탐색 범위 | Trials |
|-----------|-------------|----------|--------|
| `ce_dice` | — | — | — |
| `wce_dice` | — | — | — |
| `lwce_dice` | — | — | — |
| `plwce_dice` | alpha | 2.5 ~ 15.0 | 30 |
| `pwce_dice` | alpha | 0.2 ~ 2.5 | 30 |
| `cb_dice` | — | — | — |
| `plwce_focal_dice` | alpha + gamma | alpha 2.5~15.0, gamma 0.5~5.0 | 60 |

## 7. SoTA 참고 (2026년 3월 기준)
| 방법 | mDice (RV/MYO/LV) | 출처 |
|------|-------------------|------|
| SwinUNet (2021) | 90.00% (86.21/85.47/95.18) | ECCV'22 |
| TransUNet (2021) | 89.07% | arXiv |
| U-Net baseline | ~85~88% | 복수 논문 |

> 본 연구 목표: U-Net baseline 대비 LWCE/PLWCE 계열 손실 함수의 개선 효과 검증.
> 평가 지표: 클래스별 Dice (RV, MYO, LV), mDice (BG 제외)
> 결과 저장: `medical_data/results/ACDC_Cardiac_MRI/`

In [ ]:
# === Cell 0: 환경설정 ===
import os, sys, random, json, warnings
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import cv2
import zipfile
import shutil
from pathlib import Path
from sklearn.model_selection import train_test_split

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import segmentation_models_pytorch as smp

import optuna
from tqdm.auto import tqdm

warnings.filterwarnings('ignore')

# --- nibabel (NIfTI 로드) ---
try:
    import nibabel as nib
except ImportError:
    import subprocess; subprocess.run(['pip', 'install', 'nibabel', '-q'])
    import nibabel as nib

# --- custom_losses 경로 ---
_CL_LOCAL = '/root/imbalanced-data-LWCE/medical_data'
_CL_COLAB = '/tmp/custom_losses'
if os.path.exists(_CL_LOCAL):
    sys.path.insert(0, _CL_LOCAL)
else:
    sys.path.insert(0, _CL_COLAB)
from custom_losses import get_loss_function, calculate_weights

# --- 디바이스 ---
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

# --- 시드 고정 ---
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# --- Google Drive 마운트 ---
GDRIVE_ZIP = '/content/drive/MyDrive/imbalanced-data-LWCE/acdc/acdc.zip'
IS_COLAB = False
try:
    from google.colab import drive
    drive.mount('/content/drive')
    IS_COLAB = True
    # custom_losses.py Colab 복사
    _cl_src = '/content/drive/MyDrive/imbalanced-data-LWCE/medical_data/custom_losses.py'
    if not os.path.exists(os.path.join(_CL_LOCAL, 'custom_losses.py')) and os.path.exists(_cl_src):
        os.makedirs(_CL_COLAB, exist_ok=True)
        shutil.copy(_cl_src, _CL_COLAB)
    print('Google Drive 마운트 완료')
except Exception:
    print('Colab 환경 아님 — 로컬/수동 경로 사용')

# --- 결과 저장 경로 ---
RESULTS_DIR = '/root/imbalanced-data-LWCE/medical_data/results/ACDC_Cardiac_MRI'
os.makedirs(RESULTS_DIR, exist_ok=True)

# --- 하이퍼파라미터 ---
IMG_SIZE      = 256
BATCH_SIZE    = 16
NUM_WORKERS   = 2
FINAL_EPOCHS  = 100
FINAL_LR      = 1e-4
PROXY_EPOCHS  = 10
PROXY_SUBSET  = 0.15
N_TRIALS      = 30
N_TRIALS_PF   = 60

NUM_CLASSES   = 4
CLASS_NAMES   = ['BG', 'RV', 'MYO', 'LV']

IMAGENET_MEAN = np.array([0.485, 0.456, 0.406], dtype=np.float32)
IMAGENET_STD  = np.array([0.229, 0.224, 0.225], dtype=np.float32)

print('환경설정 완료')

In [ ]:
# === Cell 1: 데이터 ===

# --- zip 압축 해제 ---
DATA_DIR     = '/tmp/acdc_data'
CHECK_SUBDIR = 'training'  # 압축 해제 완료 여부 확인

if not os.path.exists(os.path.join(DATA_DIR, CHECK_SUBDIR)):
    if IS_COLAB and os.path.exists(GDRIVE_ZIP):
        print(f'Drive에서 압축 해제 중: {GDRIVE_ZIP}')
        os.makedirs(DATA_DIR, exist_ok=True)
        with zipfile.ZipFile(GDRIVE_ZIP, 'r') as zf:
            zf.extractall(DATA_DIR)
        # zip 내부 최상위 폴더가 acdc/ 인 경우 한 단계 올리기
        _inner = os.path.join(DATA_DIR, 'acdc')
        if os.path.isdir(_inner):
            for item in os.listdir(_inner):
                shutil.move(os.path.join(_inner, item), DATA_DIR)
            os.rmdir(_inner)
        print('압축 해제 완료')
    else:
        print('[데이터 없음] Drive에 acdc.zip 업로드 또는 /tmp/acdc_data/training/에 직접 배치')

TRAINING_DIR = os.path.join(DATA_DIR, 'training')
print(f'데이터 경로: {TRAINING_DIR}')

# --- Info_*.cfg 파싱: ED/ES 프레임 번호 추출 ---
def parse_cfg(cfg_path):
    info = {}
    with open(cfg_path, 'r') as f:
        for line in f:
            if ':' in line:
                k, v = line.strip().split(':', 1)
                info[k.strip()] = v.strip()
    return int(info['ED']), int(info['ES'])

# --- 볼륨 단위 percentile 정규화 ---
def percentile_normalize_volume(vol):
    """비-zero 픽셀 기준 1~99th percentile 정규화"""
    mask = vol > 0
    if mask.sum() == 0:
        return vol.astype(np.float32)
    vals = vol[mask]
    lo = np.percentile(vals, 1)
    hi = np.percentile(vals, 99)
    vol = np.clip(vol, lo, hi)
    return ((vol - lo) / (hi - lo + 1e-8)).astype(np.float32)

# --- 환자 폴더에서 ED/ES 슬라이스 추출 ---
def load_patient_slices(patient_dir):
    """(img_2d, mask_2d) 튜플 리스트 반환. 정규화는 볼륨 단위로 수행."""
    patient_id = os.path.basename(patient_dir)
    cfg_path   = os.path.join(patient_dir, f'Info_{patient_id}.cfg')
    if not os.path.exists(cfg_path):
        return []

    ed_frame, es_frame = parse_cfg(cfg_path)
    slices = []

    for frame_idx in [ed_frame, es_frame]:
        img_path = os.path.join(patient_dir, f'{patient_id}_frame{frame_idx:02d}.nii.gz')
        gt_path  = os.path.join(patient_dir, f'{patient_id}_frame{frame_idx:02d}_gt.nii.gz')
        if not (os.path.exists(img_path) and os.path.exists(gt_path)):
            continue

        img_vol = nib.load(img_path).get_fdata().astype(np.float32)   # (H, W, D)
        gt_vol  = nib.load(gt_path).get_fdata().astype(np.int64)      # (H, W, D)

        img_vol = percentile_normalize_volume(img_vol)

        for s in range(img_vol.shape[2]):
            slices.append((img_vol[:, :, s], gt_vol[:, :, s]))

    return slices

# --- 환자 단위 분할 (data leakage 방지) ---
patient_dirs = sorted([
    os.path.join(TRAINING_DIR, d)
    for d in os.listdir(TRAINING_DIR)
    if os.path.isdir(os.path.join(TRAINING_DIR, d)) and d.startswith('patient')
])
patient_ids = [os.path.basename(d) for d in patient_dirs]
print(f'총 환자 수: {len(patient_ids)}')

tr_ids, tmp_ids   = train_test_split(patient_ids, test_size=0.30, random_state=SEED)
val_ids, test_ids = train_test_split(tmp_ids,      test_size=0.50, random_state=SEED)
print(f'Train: {len(tr_ids)}, Val: {len(val_ids)}, Test: {len(test_ids)} patients')

# --- 슬라이스 수집 ---
def collect_slices(id_list, desc):
    all_slices = []
    for pid in tqdm(id_list, desc=desc):
        all_slices.extend(load_patient_slices(os.path.join(TRAINING_DIR, pid)))
    return all_slices

tr_slices   = collect_slices(tr_ids,   '슬라이스 수집 (Train)')
val_slices  = collect_slices(val_ids,  '슬라이스 수집 (Val)')
test_slices = collect_slices(test_ids, '슬라이스 수집 (Test)')
print(f'슬라이스 수 — Train: {len(tr_slices)}, Val: {len(val_slices)}, Test: {len(test_slices)}')

# --- Dataset ---
class ACDCDataset(Dataset):
    def __init__(self, slices, augment=False):
        self.slices  = slices
        self.augment = augment

    def __len__(self):
        return len(self.slices)

    def __getitem__(self, idx):
        img, mask = self.slices[idx]

        # 256×256 리사이즈
        img_r  = cv2.resize(img,                              (IMG_SIZE, IMG_SIZE), interpolation=cv2.INTER_LINEAR)
        mask_r = cv2.resize(mask.astype(np.uint8),            (IMG_SIZE, IMG_SIZE), interpolation=cv2.INTER_NEAREST).astype(np.int64)

        # Grayscale → 3채널 복제 → ImageNet 정규화
        img_3ch = np.stack([img_r, img_r, img_r], axis=2)    # (H, W, 3)
        img_3ch = (img_3ch - IMAGENET_MEAN) / IMAGENET_STD

        # 증강
        if self.augment:
            if random.random() > 0.5:
                img_3ch = np.fliplr(img_3ch).copy()
                mask_r  = np.fliplr(mask_r).copy()
            if random.random() > 0.5:
                img_3ch = np.flipud(img_3ch).copy()
                mask_r  = np.flipud(mask_r).copy()
            k = random.randint(0, 3)
            if k > 0:
                img_3ch = np.rot90(img_3ch, k).copy()
                mask_r  = np.rot90(mask_r,  k).copy()

        img_t  = torch.from_numpy(img_3ch.transpose(2, 0, 1)).float()  # (3, H, W)
        mask_t = torch.from_numpy(mask_r).long()                        # (H, W)
        return img_t, mask_t

train_ds   = ACDCDataset(tr_slices,   augment=True)
val_ds     = ACDCDataset(val_slices,  augment=False)
test_ds    = ACDCDataset(test_slices, augment=False)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=NUM_WORKERS, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)

# --- 클래스 비율 계산 (학습 데이터 기준, 리사이즈 후 픽셀 단위) ---
class_counts = np.zeros(NUM_CLASSES, dtype=np.int64)
for _, mask_t in tqdm(train_loader, desc='클래스 비율 계산'):
    for c in range(NUM_CLASSES):
        class_counts[c] += int((mask_t == c).sum())

total = class_counts.sum()
print('\n클래스 비율:')
for name, count in zip(CLASS_NAMES, class_counts):
    print(f'  {name}: {count:,} ({count / total * 100:.2f}%)')
for c in range(1, NUM_CLASSES):
    print(f'  BG:{CLASS_NAMES[c]} = {class_counts[0] / class_counts[c]:.1f}:1')

In [ ]:
# === Cell 2: 모델 ===

# --- U-Net (ResNet34) — 4-class 출력 ---
def build_model():
    return smp.Unet(
        encoder_name    = 'resnet34',
        encoder_weights = 'imagenet',
        in_channels     = 3,
        classes         = NUM_CLASSES,   # 4-class: 2ch 변환 불필요
        activation      = None,
    ).to(device)

# --- 검증 지표: 클래스별 Dice (RV/MYO/LV) + mDice ---
def compute_val_metrics(model, loader):
    model.eval()
    dice_sum  = np.zeros(NUM_CLASSES - 1)  # BG 제외
    n_batches = 0

    with torch.no_grad():
        for imgs, masks in loader:
            imgs  = imgs.to(device)
            masks = masks.to(device)
            preds = model(imgs).argmax(dim=1)  # (B, H, W)

            for c_idx, c in enumerate(range(1, NUM_CLASSES)):
                p = (preds == c).float()
                t = (masks == c).float()
                inter = (p * t).sum().item()
                union = (p.sum() + t.sum()).item()
                if union > 0:
                    dice_sum[c_idx] += 2. * inter / (union + 1e-8)

            n_batches += 1

    dice_per_class = dice_sum / max(n_batches, 1)
    mdice = float(dice_per_class.mean())
    return {
        'mDice':   mdice,
        'Dice_RV':  float(dice_per_class[0]),
        'Dice_MYO': float(dice_per_class[1]),
        'Dice_LV':  float(dice_per_class[2]),
    }

In [ ]:
# === Cell 3: 학습함수 ===

def train_model(loss_name, alpha=1.0, gamma=2.0, epochs=50, lr=1e-4,
                subset_ratio=1.0, tag=''):
    """
    ACDC 4-class 세그멘테이션 학습.
    multi-class이므로 to_2ch_logits 변환 없이 직접 criterion에 전달.
    """
    model     = build_model()
    optimizer = optim.Adam(model.parameters(), lr=lr, weight_decay=1e-5)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)
    criterion = get_loss_function(loss_name, class_counts=class_counts, alpha=alpha, gamma=gamma)

    # --- 서브셋 로더 (Optuna proxy) ---
    if subset_ratio < 1.0:
        n      = max(1, int(len(train_ds) * subset_ratio))
        sub_ds = torch.utils.data.Subset(train_ds, random.sample(range(len(train_ds)), n))
        loader = DataLoader(sub_ds, batch_size=BATCH_SIZE, shuffle=True,
                            num_workers=NUM_WORKERS, pin_memory=True)
    else:
        loader = train_loader

    best_mdice = 0.0
    best_state = None
    history    = {'loss': [], 'val_mdice': []}
    ckpt_path  = f'/tmp/best_acdc_{tag}_{loss_name}.pth'

    for epoch in range(epochs):
        model.train()
        epoch_loss = 0.0

        for imgs, masks in tqdm(loader,
                                desc=f'[{tag}] {loss_name} Ep{epoch+1:02d}/{epochs}',
                                leave=False):
            imgs  = imgs.to(device)
            masks = masks.to(device)

            optimizer.zero_grad()
            logits = model(imgs)          # (B, 4, H, W)
            loss   = criterion(logits, masks)
            loss.backward()
            optimizer.step()
            epoch_loss += loss.item()

        scheduler.step()

        val_metrics = compute_val_metrics(model, val_loader)
        val_mdice   = val_metrics['mDice']
        history['loss'].append(epoch_loss / len(loader))
        history['val_mdice'].append(val_mdice)

        if val_mdice > best_mdice:
            best_mdice = val_mdice
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            torch.save(best_state, ckpt_path)

    if best_state is not None:
        model.load_state_dict(best_state)

    return model, history, best_mdice

In [ ]:
# === Cell 4: Optuna ===
os.environ['TQDM_DISABLE'] = '1'

# --- 탐색 범위 ---
ALPHA_LOW_PLWCE  = 2.5;  ALPHA_HIGH_PLWCE = 15.0
ALPHA_LOW_PWCE   = 0.2;  ALPHA_HIGH_PWCE  = 2.5
GAMMA_LOW        = 0.5;  GAMMA_HIGH       = 5.0

def make_objective(loss_name, alpha_low, alpha_high, gamma_low=None, gamma_high=None):
    def objective(trial):
        alpha = trial.suggest_float('alpha', alpha_low, alpha_high)
        gamma = trial.suggest_float('gamma', gamma_low, gamma_high) if gamma_low is not None else 2.0
        try:
            _, _, mdice = train_model(
                loss_name    = loss_name,
                alpha        = alpha,
                gamma        = gamma,
                epochs       = PROXY_EPOCHS,
                subset_ratio = PROXY_SUBSET,
                tag          = f'trial{trial.number}',
            )
            return mdice
        except Exception as e:
            print(f'Trial {trial.number} 실패: {e}')
            return 0.0
    return objective

# --- PLWCE alpha 탐색 ---
print('=== PLWCE alpha 탐색 ===')
study_plwce = optuna.create_study(
    direction='maximize',
    pruner=optuna.pruners.MedianPruner(n_startup_trials=5, n_warmup_steps=2)
)
study_plwce.optimize(make_objective('plwce_dice', ALPHA_LOW_PLWCE, ALPHA_HIGH_PLWCE),
                     n_trials=N_TRIALS, show_progress_bar=False)
best_alpha_plwce = study_plwce.best_params['alpha']
print(f'PLWCE best alpha: {best_alpha_plwce:.4f}  mDice: {study_plwce.best_value:.4f}')

# --- PWCE alpha 탐색 ---
print('=== PWCE alpha 탐색 ===')
study_pwce = optuna.create_study(
    direction='maximize',
    pruner=optuna.pruners.MedianPruner(n_startup_trials=5, n_warmup_steps=2)
)
study_pwce.optimize(make_objective('pwce_dice', ALPHA_LOW_PWCE, ALPHA_HIGH_PWCE),
                    n_trials=N_TRIALS, show_progress_bar=False)
best_alpha_pwce = study_pwce.best_params['alpha']
print(f'PWCE best alpha: {best_alpha_pwce:.4f}  mDice: {study_pwce.best_value:.4f}')

# --- PLWCE+Focal alpha+gamma 탐색 ---
print('=== PLWCE+Focal alpha+gamma 탐색 ===')
study_pf = optuna.create_study(
    direction='maximize',
    pruner=optuna.pruners.MedianPruner(n_startup_trials=5, n_warmup_steps=2)
)
study_pf.optimize(
    make_objective('plwce_focal_dice', ALPHA_LOW_PLWCE, ALPHA_HIGH_PLWCE, GAMMA_LOW, GAMMA_HIGH),
    n_trials=N_TRIALS_PF, show_progress_bar=False
)
best_alpha_pf = study_pf.best_params['alpha']
best_gamma_pf = study_pf.best_params['gamma']
print(f'PLWCE+Focal best alpha: {best_alpha_pf:.4f}, gamma: {best_gamma_pf:.4f}  mDice: {study_pf.best_value:.4f}')

os.environ.pop('TQDM_DISABLE', None)

# --- 탐색 시각화 (study 정의 이후) ---
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle('ACDC Cardiac MRI — Optuna Alpha/Gamma 탐색 결과')

for ax, study, name in [
    (axes[0], study_plwce, 'PLWCE'),
    (axes[1], study_pwce,  'PWCE'),
]:
    trials = [t for t in study.trials if t.value is not None]
    xs     = [t.params['alpha'] for t in trials]
    ys     = [t.value for t in trials]
    ax.scatter(xs, ys, alpha=0.6, s=40, color='steelblue')
    bx = study.best_params['alpha']
    by = study.best_value
    ax.axvline(bx, color='red', linestyle='--', linewidth=1.5, label=f'Best alpha={bx:.3f}')
    ax.scatter([bx], [by], color='red', s=100, zorder=5)
    ax.set_xlabel('alpha'); ax.set_ylabel('Val mDice')
    ax.set_title(f'{name} alpha 탐색'); ax.legend(); ax.grid(True)

ax = axes[2]
trials_pf = [t for t in study_pf.trials if t.value is not None]
alphas_pf = [t.params['alpha'] for t in trials_pf]
gammas_pf = [t.params['gamma'] for t in trials_pf]
values_pf = [t.value for t in trials_pf]
sc = ax.scatter(alphas_pf, gammas_pf, c=values_pf, cmap='viridis', alpha=0.7, s=40)
ax.scatter([best_alpha_pf], [best_gamma_pf], color='red', s=150, zorder=5,
           marker='*', label=f'Best α={best_alpha_pf:.3f}, γ={best_gamma_pf:.3f}')
plt.colorbar(sc, ax=ax, label='Val mDice')
ax.set_xlabel('alpha'); ax.set_ylabel('gamma')
ax.set_title('PLWCE+Focal alpha+gamma 탐색'); ax.legend(); ax.grid(True)

plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, 'ACDC_optuna_search.png'), dpi=100)
plt.close()
print(f'탐색 결과 이미지 저장: {RESULTS_DIR}/ACDC_optuna_search.png')

In [ ]:
# === Cell 5: 학습실행 ===

experiments = [
    ('ce_dice',          1.0,              2.0,           'CE+Dice (baseline)'),
    ('wce_dice',         1.0,              2.0,           'WCE+Dice'),
    ('lwce_dice',        1.0,              2.0,           'LWCE+Dice'),
    ('plwce_dice',       best_alpha_plwce, 2.0,           f'PLWCE+Dice (α={best_alpha_plwce:.2f})'),
    ('pwce_dice',        best_alpha_pwce,  2.0,           f'PWCE+Dice (α={best_alpha_pwce:.2f})'),
    ('cb_dice',          1.0,              2.0,           'CB+Dice'),
    ('plwce_focal_dice', best_alpha_pf,    best_gamma_pf, f'PLWCE+Focal+Dice (α={best_alpha_pf:.2f}, γ={best_gamma_pf:.2f})'),
]

all_results = {}
for loss_name, alpha, gamma, label in experiments:
    print(f'\n{"="*60}')
    print(f'학습: {label}')
    print(f'{"="*60}')
    model, history, best_mdice = train_model(
        loss_name    = loss_name,
        alpha        = alpha,
        gamma        = gamma,
        epochs       = FINAL_EPOCHS,
        lr           = FINAL_LR,
        tag          = 'final',
    )
    all_results[label] = {
        'model':          model,
        'history':        history,
        'best_val_mdice': best_mdice,
        'loss_name':      loss_name,
        'alpha':          alpha,
        'gamma':          gamma,
    }
    print(f'  Best Val mDice: {best_mdice:.4f}')

print('\n모든 학습 완료!')

In [ ]:
# === Cell 6: 평가/저장 ===

# --- Test set 최종 평가 ---
print('=== Test Set 최종 평가 ===')
final_results = {}
for label, v in all_results.items():
    metrics = compute_val_metrics(v['model'], test_loader)
    final_results[label] = {
        'loss_name':      v['loss_name'],
        'alpha':          v['alpha'],
        'gamma':          v['gamma'],
        'best_val_mdice': v['best_val_mdice'],
        **metrics,
    }
    print(f'{label}: mDice={metrics["mDice"]:.4f}  '
          f'RV={metrics["Dice_RV"]:.4f}  MYO={metrics["Dice_MYO"]:.4f}  LV={metrics["Dice_LV"]:.4f}')

# --- 학습 곡선 ---
n_exp  = len(all_results)
n_cols = 4
n_rows = (n_exp + n_cols - 1) // n_cols
fig, axes = plt.subplots(n_rows, n_cols, figsize=(n_cols * 5, n_rows * 4))
axes = axes.flatten()
fig.suptitle('ACDC Cardiac MRI — 학습 곡선 (Loss & Val mDice)')

for i, (label, v) in enumerate(all_results.items()):
    ax  = axes[i]
    ax2 = ax.twinx()
    ep  = range(1, len(v['history']['loss']) + 1)
    ax.plot(ep,  v['history']['loss'],     'b-', alpha=0.7, label='Train Loss')
    ax2.plot(ep, v['history']['val_mdice'], 'r-', alpha=0.7, label='Val mDice')
    ax.set_xlabel('Epoch')
    ax.set_ylabel('Loss', color='b')
    ax2.set_ylabel('Val mDice', color='r')
    ax.set_title(label, fontsize=9)
    ax.legend(loc='upper left', fontsize=7)
    ax2.legend(loc='upper right', fontsize=7)
    ax.grid(True, alpha=0.3)

for j in range(i + 1, len(axes)):
    axes[j].set_visible(False)

plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, 'ACDC_training_curves.png'), dpi=100)
plt.close()
print(f'학습 곡선 저장: {RESULTS_DIR}/ACDC_training_curves.png')

# --- 예측 시각화 ---
best_label = max(final_results, key=lambda k: final_results[k]['mDice'])
best_model = all_results[best_label]['model']
best_model.eval()

# 4-class 컬러맵: BG=흑, RV=적, MYO=녹, LV=청
_CMAP = np.array([[0,0,0],[255,0,0],[0,200,0],[0,0,255]], dtype=np.uint8)

def colorize_mask(mask_np):
    rgb = np.zeros((*mask_np.shape, 3), dtype=np.uint8)
    for c, color in enumerate(_CMAP):
        rgb[mask_np == c] = color
    return rgb

sample_imgs, sample_masks = next(iter(test_loader))
with torch.no_grad():
    sample_preds = best_model(sample_imgs.to(device)).argmax(dim=1).cpu()

n_show = min(4, len(sample_imgs))
fig, axes = plt.subplots(n_show, 3, figsize=(12, 4 * n_show))
if n_show == 1:
    axes = axes[np.newaxis, :]
fig.suptitle(f'ACDC 예측 시각화 (Best: {best_label})\nBlack=BG  Red=RV  Green=MYO  Blue=LV')

for i in range(n_show):
    img_np  = sample_imgs[i, 0].numpy()
    gt_np   = sample_masks[i].numpy()
    pred_np = sample_preds[i].numpy()
    axes[i, 0].imshow(img_np,              cmap='gray'); axes[i, 0].set_title('Input (gray)'); axes[i, 0].axis('off')
    axes[i, 1].imshow(colorize_mask(gt_np));              axes[i, 1].set_title('Ground Truth'); axes[i, 1].axis('off')
    axes[i, 2].imshow(colorize_mask(pred_np));            axes[i, 2].set_title('Prediction');   axes[i, 2].axis('off')

plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, 'ACDC_prediction_vis.png'), dpi=100)
plt.close()
print(f'예측 시각화 저장: {RESULTS_DIR}/ACDC_prediction_vis.png')

# --- 최종 지표 바차트 ---
labels_plot = list(final_results.keys())
short_labels = [k.split('(')[0].strip() for k in labels_plot]
mdice_v  = [final_results[k]['mDice']   for k in labels_plot]
rv_v     = [final_results[k]['Dice_RV'] for k in labels_plot]
myo_v    = [final_results[k]['Dice_MYO'] for k in labels_plot]
lv_v     = [final_results[k]['Dice_LV'] for k in labels_plot]

x = np.arange(len(labels_plot))
w = 0.2
fig, ax = plt.subplots(figsize=(16, 6))
ax.bar(x - 1.5*w, mdice_v, w, label='mDice',    color='steelblue',      alpha=0.85)
ax.bar(x - 0.5*w, rv_v,    w, label='Dice_RV',  color='tomato',         alpha=0.85)
ax.bar(x + 0.5*w, myo_v,   w, label='Dice_MYO', color='mediumseagreen', alpha=0.85)
ax.bar(x + 1.5*w, lv_v,    w, label='Dice_LV',  color='mediumpurple',   alpha=0.85)
ax.set_xticks(x)
ax.set_xticklabels(short_labels, rotation=20, ha='right')
ax.set_ylabel('Dice Score')
ax.set_title('ACDC Cardiac MRI — Loss별 최종 성능 비교')
ax.legend(); ax.grid(True, axis='y', alpha=0.3); ax.set_ylim(0, 1.05)
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, 'ACDC_final_metrics.png'), dpi=100)
plt.close()
print(f'최종 지표 바차트 저장: {RESULTS_DIR}/ACDC_final_metrics.png')

# --- JSON 저장 ---
save_data = {
    'domain':      'ACDC Cardiac MRI Segmentation',
    'model':       'U-Net (ResNet34, ImageNet pretrained)',
    'num_classes': NUM_CLASSES,
    'class_names': CLASS_NAMES,
    'class_counts': {n: int(c) for n, c in zip(CLASS_NAMES, class_counts)},
    'imbalance': {
        f'BG:{CLASS_NAMES[c]}': round(float(class_counts[0]) / float(class_counts[c]), 1)
        if class_counts[c] > 0 else None
        for c in range(1, NUM_CLASSES)
    },
    'optuna': {
        'best_alpha_plwce': best_alpha_plwce,
        'best_alpha_pwce':  best_alpha_pwce,
        'best_alpha_pf':    best_alpha_pf,
        'best_gamma_pf':    best_gamma_pf,
    },
    'results': {
        k: {mk: float(mv) if isinstance(mv, (float, np.floating)) else mv
            for mk, mv in v.items()}
        for k, v in final_results.items()
    },
    'best_model': max(final_results, key=lambda k: final_results[k]['mDice']),
}
json_path = os.path.join(RESULTS_DIR, 'ACDC_final_results.json')
with open(json_path, 'w', encoding='utf-8') as f:
    json.dump(save_data, f, indent=2, ensure_ascii=False)
print(f'JSON 저장: {json_path}')

# --- Excel 저장 ---
summary_rows = []
for label, v in final_results.items():
    summary_rows.append({
        'Loss Function': label,
        'alpha':         round(v['alpha'], 4),
        'gamma':         round(v['gamma'], 4),
        'mDice':         round(v['mDice'],    4),
        'Dice_RV':       round(v['Dice_RV'],  4),
        'Dice_MYO':      round(v['Dice_MYO'], 4),
        'Dice_LV':       round(v['Dice_LV'],  4),
        'Best_Val_mDice': round(v['best_val_mdice'], 4),
    })

history_rows = []
for label, v in all_results.items():
    for ep, (loss_val, mdice_val) in enumerate(
            zip(v['history']['loss'], v['history']['val_mdice']), 1):
        history_rows.append({
            'Loss Function': label,
            'Epoch':         ep,
            'Train Loss':    round(loss_val,  6),
            'Val mDice':     round(mdice_val, 6),
        })

df_summary = pd.DataFrame(summary_rows)
df_history = pd.DataFrame(history_rows)

excel_path = os.path.join(RESULTS_DIR, 'ACDC_final_results.xlsx')
with pd.ExcelWriter(excel_path, engine='openpyxl') as writer:
    df_summary.to_excel(writer, sheet_name='Summary',          index=False)
    df_history.to_excel(writer, sheet_name='Training_History', index=False)
print(f'Excel 저장: {excel_path}')

# --- 최종 요약 출력 ---
print('\n=== 최종 결과 요약 ===')
print(df_summary.to_string(index=False))
print(f'\n최고 성능 모델: {save_data["best_model"]}')